# The marker game

Welcome. Here is the situation.

A hospital ran a study of **coronary artery disease**: 600 patients and 600 people without the disease.
Everyone gave a blood sample, and the lab measured the markers you know from the lecture (LDL
cholesterol, ApoB, Lp(a), HDL, CRP, troponin, NT-proBNP), plus **200 other proteins** from a large
protein panel. The hospital now wants to know which of those 200 proteins matter.

That is your job today: find the proteins that matter, work out *why* each one is linked to the
disease, and decide what each one is good for: a diagnostic panel, a list for the lab to follow up,
or a drug target.

**Why the 200 proteins have no names.** In a real study you would know every protein's name, and the
first thing anyone does is look it up. We hid the names on purpose. Today, the only way to learn what a
protein does is to ask the data, which is exactly the skill this session is about. At the end you get
the names, the stories, and a score.

## How the session works

There are 10 tasks in 6 parts. Each task asks you to **decide** something, run it, and say what it
means. Where you see **Task**, the cell below it is yours: the comment in it gives you something to
think about and the function to use. At the end of each part there is a cell for **1 sentence of
conclusion** in your own words.

Along the way you will build up **your verdicts**: for each protein you believe matters, what kind of
link it has to the disease and what it is good for. The last task collects them, and then we reveal the
answers and count your score.

Everything you need is in this notebook, including the toolkit. Each function sits in its own cell,
with a short explanation, so you can read what it does.

**In Google Colab:** use a Python CPU runtime and run the setup cells first. The CSV loads directly from GitHub. Use **File → Save a copy in Drive** to keep your work.

[Lab handout](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-06-feature-selection/notebooks/session_06/FS_lab_handout.md) · [Data dictionary](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-06-feature-selection/notebooks/session_06/data/cad_lab_DICTIONARY.md) · [Study CSV](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-06-feature-selection/notebooks/session_06/data/cad_lab_study.csv).


## Setup

Run the cells in this section. The first installs the packages; the next loads the data.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import itertools
import warnings
warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 40)

df = pd.read_csv("https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-06-feature-selection/notebooks/session_06/data/cad_lab_study.csv")

MARKERS = ["ldl_c", "apob", "lpa", "hdl", "crp", "troponin", "ntprobnp", "hemolysis", "statin", "bmi", "smoking", "age", "female"]
TARGET = "cad"
PROTEINS = [c for c in df.columns if c.startswith("protein_")]
ALL = MARKERS + PROTEINS

print("""Number of people:    {people}
Number of patients:  {patients}
Named markers:       {markers}
Unnamed proteins:    {proteins}
Cohorts:             {cohorts}""".format(people=len(df), patients=int(df.cad.sum()), markers=len(MARKERS), proteins=len(PROTEINS),
           cohorts=", ".join(f"{k} ({v})" for k, v in df.cohort.value_counts().items())))

A first look at the table. The label is `cad`: 1 means the person has diagnosed coronary artery disease.

In [ ]:
df.head()

## The toolkit

Each function below does 1 job. Read the sentence above each one: it tells you what the function
answers and how to read its output. You do not need to understand every line of code, but it is all
here, so nothing is a black box.

**A helper.** Some measurements (troponin, CRP, Lp(a) and others) are very skewed: most values are small and a few are huge. Before any method sees them we take the logarithm of such columns and put every column on the same scale. This is standard, and it is done inside every function.

In [ ]:
def prep(df, features):
    """Log-transform skewed positive columns, then standardise. Returns a numpy array."""
    X = df[features].astype(float).copy()
    for c in features:
        v = X[c]
        if v.min() > 0 and stats.skew(v) > 1: X[c] = np.log(v)
    return StandardScaler().fit_transform(X)

**How well does one feature separate patients from controls on its own?** The answer is an AUC, as in Session 4: 0.5 means not at all, 1.0 means perfectly, and 0.0 *also* means perfectly, just the other way round (the feature is *lower* in patients). A feature at 0.1 is as strong as one at 0.9.

In [ ]:
def auc_alone(df, feature):
    """How well one feature separates patients from controls on its own (AUC; 0.5 = nothing)."""
    return round(float(roc_auc_score(df[TARGET], df[feature])), 3)

**And a set of features together?** This fits a logistic model and scores it with cross-validation: 5 folds, each person scored by a model that did not see them, as in Session 5.

In [ ]:
def _cv(): return StratifiedKFold(5, shuffle=True, random_state=0)

def auc_together(df, features):
    """Cross-validated AUC of a logistic model on these features."""
    X = prep(df, features); y = df[TARGET].values
    p = cross_val_predict(LogisticRegression(max_iter=3000), X, y, cv=_cv(), method="predict_proba")[:, 1]
    return round(float(roc_auc_score(y, p)), 3)

**Rank everything.** For every feature this gives you 4 things:

- `auc`: the feature alone, as above;
- `p`: a *p-value*: if this feature had nothing to do with the disease, how likely is a difference this large between patients and controls, just by chance? Small means unlikely;
- `q`: the p-value corrected for the fact that you are testing 100s of features at once (we explain this in part B);
- `passes`: whether `q` is at most 0.05.

In [ ]:
def rank(df, features, q=0.05):
    """Every feature on its own: AUC, the p-value of a t-test, and its false-discovery-controlled
    q-value (Benjamini-Hochberg). Sorted by p-value. The column 'passes' marks q <= 0.05."""
    X = prep(df, features); y = df[TARGET].values
    pv = np.array([stats.ttest_ind(X[y == 1, j], X[y == 0, j], equal_var=False).pvalue for j in range(len(features))])
    order = np.argsort(pv); m = len(pv); qv = np.empty(m); prev = 1.0
    for r in range(m - 1, -1, -1):
        j = order[r]; prev = min(prev, pv[j] * m / (r + 1)); qv[j] = prev
    out = pd.DataFrame({"feature": features, "auc": [auc_alone(df, f) for f in features], "p": pv, "q": qv})
    out["passes"] = out.q <= q
    return out.sort_values("p").reset_index(drop=True)
    # auc: how well the feature alone separates patients from controls (0.5 = not at all, 1 or 0 = perfectly)
    # p:   the chance of a difference this large between patients and controls if the feature had nothing to do with the disease
    # q:   the p-value corrected for testing many features at once - see the notebook for what it means
    # passes: q is at most 0.05

**1 method, 1 list.** `select` runs a selection method and returns the features it picks. The methods are the ones from the lecture:

- `filter`: the ranking with the correction, keeping what passes;
- `mrmr`: picks features one at a time, rewarding relevance and punishing redundancy;
- `lasso` and `elastic_net`: selection inside the model fit;
- `rfe`: fit, drop the weakest feature, repeat;
- `boruta`: every feature competes against a shuffled copy of itself;
- `stability`: keeps what comes back in at least 60% of random halves of the data (the next cell explains it);
- `markov`: the conditional-independence search explained in part C.

In [ ]:
def select(df, method, features, k=10):
    """Run one selection method and return the list it picks.
    method: 'filter' (t-test + FDR 5%), 'mrmr', 'lasso', 'elastic_net', 'rfe', 'boruta', 'stability', 'markov'."""
    X = prep(df, features); y = df[TARGET].values; p = X.shape[1]
    if method == "filter":
        r = rank(df, features); return list(r[r.passes].feature)
    if method == "mrmr":
        rel = np.abs([np.corrcoef(X[:, j], y)[0, 1] for j in range(p)]); C = np.abs(np.corrcoef(X.T)); sel = [int(np.argmax(rel))]
        while len(sel) < k: sel.append(int(np.argmax([rel[j] - C[j, sel].mean() if j not in sel else -9 for j in range(p)])))
        return [features[j] for j in sel]
    if method == "lasso":
        m = LogisticRegressionCV(penalty="l1", solver="saga", Cs=12, cv=_cv(), max_iter=5000, scoring="roc_auc").fit(X, y); return [features[j] for j in np.flatnonzero(m.coef_[0])]
    if method == "elastic_net":
        m = LogisticRegressionCV(penalty="elasticnet", solver="saga", l1_ratios=[0.5], Cs=12, cv=_cv(), max_iter=5000, scoring="roc_auc").fit(X, y); return [features[j] for j in np.flatnonzero(m.coef_[0])]
    if method == "rfe":
        m = RFE(LogisticRegression(max_iter=3000), n_features_to_select=k, step=0.2).fit(X, y); return [features[j] for j in np.flatnonzero(m.support_)]
    if method == "boruta":
        rng = np.random.default_rng(0); hits = np.zeros(p); T = 30
        for t in range(T):
            sh = rng.permuted(X, axis=0); imp = RandomForestClassifier(200, max_depth=6, n_jobs=-1, random_state=t).fit(np.c_[X, sh], y).feature_importances_
            hits += imp[:p] > imp[p:].max()
        return [features[j] for j in range(p) if hits[j] >= 0.6 * T]
    if method == "stability":
        f = stability(df, features); return list(f[f >= 0.6].index)
    if method == "markov":
        return markov_boundary(df, features)[0]
    raise ValueError(method)

**Would you find it again?** Stability selection re-runs a selection method (the Lasso, from the list above) on 100 random halves of the data and counts how often each feature is chosen. A real signal comes back most of the time; a lucky one does not.

In [ ]:
def stability(df, features, C=0.1, n=100, seed=0):
    """Selection frequency of each feature over n Lasso fits on half-samples."""
    X = prep(df, features); y = df[TARGET].values; rng = np.random.default_rng(seed); freq = np.zeros(len(features))
    for _ in range(n):
        i = rng.choice(len(y), len(y) // 2, replace=False)
        freq += LogisticRegression(penalty="l1", solver="saga", C=C, max_iter=3000).fit(X[i], y[i]).coef_[0] != 0
    return pd.Series(freq / n, index=features, name="selection frequency").sort_values(ascending=False)

**Does this feature still tell you something once other features are known?** This is the question behind part C. `ci_test` gives the *partial correlation* between a feature and the disease: the correlation that is left after removing what the `given` features already explain, and its p-value. Correlation is a number from −1 to 1; 0 means no link. If you give it nothing, it is the plain correlation.

In [ ]:
def ci_test(df, feature, given=(), features=None):
    """Is the feature still associated with the disease once the 'given' features are known?
    Returns the partial correlation with the disease and its p-value (Fisher z)."""
    given = list(given); cols = [feature] + given; X = prep(df, cols); y = df[TARGET].values.astype(float); n = len(y)
    Z = np.c_[np.ones(n), X[:, 1:]] if given else np.ones((n, 1))
    ry = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]; rx = X[:, 0] - Z @ np.linalg.lstsq(Z, X[:, 0], rcond=None)[0]
    r = float(np.corrcoef(ry, rx)[0, 1]); z = np.arctanh(np.clip(r, -0.999999, 0.999999)) * np.sqrt(n - len(given) - 3)
    return pd.Series({"feature": feature, "given": ", ".join(given) if given else "nothing", "partial_correlation": round(r, 3), "p": float(2 * (1 - stats.norm.cdf(abs(z))))})

**The search.** `markov_boundary` builds the set of features that still matter once the others are known, step by step. It is explained properly in part C. It returns 2 things: the selected set, and for each selected feature the features that could *replace* it: the "statistically equivalent" ones from the lecture.

In [ ]:
def markov_boundary(df, features, alpha=0.01, max_k=2, partner_alpha=0.001):
    """A Markov-boundary search with conditional-independence tests: forward then backward for the
    parents and children (one at a time), then a partner phase - features that are nothing alone but
    dependent given the set, then a pair phase - two features that are nothing alone and something
    together, and the SES idea: for each parent or child, the features that could
    replace it. Returns (selected, equivalences)."""
    X = prep(df, features); y = df[TARGET].values.astype(float); n, p = X.shape
    def pc(j, S):
        Z = np.c_[np.ones(n), X[:, S]] if S else np.ones((n, 1))
        ry = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]; rx = X[:, j] - Z @ np.linalg.lstsq(Z, X[:, j], rcond=None)[0]
        r = np.corrcoef(ry, rx)[0, 1]; return 2 * (1 - stats.norm.cdf(abs(np.arctanh(np.clip(r, -0.999999, 0.999999)) * np.sqrt(n - len(S) - 3))))
    def maxp(j, S): return max(pc(j, list(c)) for r in range(min(max_k, len(S)) + 1) for c in itertools.combinations(S, r))
    S, cand = [], list(range(p))
    while True:
        best = min(cand, key=lambda j: maxp(j, S))
        if maxp(best, S) > alpha: break
        S.append(best); cand.remove(best)
    while True:
        worst = max(S, key=lambda j: maxp(j, [k for k in S if k != j]))
        if maxp(worst, [k for k in S if k != worst]) <= alpha: break
        S.remove(worst); cand.append(worst)
    pc_set = list(S)                                                              # parents and children
    for k in list(cand):                                                          # the partner phase: nothing alone, something given the set
        if pc(k, []) > alpha and pc(k, pc_set) < partner_alpha: S.append(k); cand.remove(k)
    for k in list(cand):                                                          # the pair phase: nothing alone, something together
        if k not in cand: continue
        for m in list(cand):
            if m != k and pc(k, S + [m]) < partner_alpha and pc(m, S + [k]) < partner_alpha:
                S += [k, m]; cand.remove(k); cand.remove(m); break
    eq = {}
    for j in pc_set:
        others = [k for k in S if k != j]
        for k in cand:
            if maxp(k, S) > alpha and maxp(k, others) <= alpha and maxp(j, others + [k]) > alpha: eq.setdefault(features[j], []).append(features[k])
    return [features[j] for j in S], eq

**Does the link hold in both cohorts?** For one feature: its AUC in the hospital cohort, its AUC in the population cohort, and a p-value for the question *does this feature depend on the cohort once we know who has the disease?* `holds` is True when the link looks the same in both.

In [ ]:
def invariance(df, feature):
    """Does the feature's link to the disease hold in both cohorts? AUC in each cohort, and the
    p-value for 'the feature depends on the cohort once the disease is known'."""
    out = {"feature": feature}
    for c in ("hospital", "population"):
        d = df[df.cohort == c]; out[f"auc_{c}"] = round(float(roc_auc_score(d[TARGET], d[feature])), 3)
    x = prep(df, [feature])[:, 0]; y = df[TARGET].values.astype(float); h = (df.cohort == "hospital").astype(float).values; n = len(y)
    Z = np.c_[np.ones(n), y, h]; beta, *_ = np.linalg.lstsq(Z, x, rcond=None); resid = x - Z @ beta
    se = np.sqrt(resid.var() * np.linalg.inv(Z.T @ Z)[2, 2]); out["p_cohort_given_disease"] = float(2 * (1 - stats.norm.cdf(abs(beta[2] / se))))
    out["holds"] = bool(out["p_cohort_given_disease"] > 0.01)
    return pd.Series(out)

**Carriers against non-carriers.** The study also took a DNA sample from everyone and read 3 genetic variants: `g_pcsk9` (a variant that lowers LDL cholesterol for life), `g_lpa` (variants that raise Lp(a)) and `g_hdl` (a variant that raises HDL cholesterol only). `genetic_test` compares the people who carry a variant with the people who do not: the marker the variant changes, the disease rate, and 2 lifestyle checks. Part E says why that comparison can tell a cause from a mere correlate.

In [ ]:
def genetic_test(df, variant, marker):
    """The Mendelian-randomization comparison: people who carry the variant against those who do
    not: the marker, the disease rate, and two lifestyle checks."""
    g = df[variant]; carriers = df[g == g.max()]; non = df[g == 0]
    row = lambda d: {marker: round(float(d[marker].mean()), 2), "disease rate": round(float(d[TARGET].mean()), 3), "bmi": round(float(d.bmi.mean()), 1), "smoking": round(float(d.smoking.mean()), 2), "n": int(len(d))}
    return pd.DataFrame({"carriers": row(carriers), "non-carriers": row(non)})

**Within patients only: does a feature differ between treated and untreated?** Patients are usually prescribed a *statin*, a cholesterol-lowering drug, when they are diagnosed. A marker that differs between patients on a statin and patients not on one is tracking the treatment, not the disease.

In [ ]:
def within_patients(df, feature, by="statin"):
    """Among the patients only: the feature's mean in each treatment group, and the p-value of the
    difference. A marker that differs by treatment within patients tracks the treatment."""
    d = df[df[TARGET] == 1]; a, b = d[d[by] == 1][feature], d[d[by] == 0][feature]
    return pd.Series({"feature": feature, f"treated ({by}=1)": round(float(a.mean()), 2), f"untreated ({by}=0)": round(float(b.mean()), 2), "p": float(stats.ttest_ind(np.log(a) if a.min() > 0 else a, np.log(b) if b.min() > 0 else b, equal_var=False).pvalue)})

In [ ]:
print("Toolkit ready.")

## A · 3 things the hospital wants

Before touching the data, let's agree on what we are looking for. The hospital wants 3 different
things, and: this was the point of the lecture: they are 3 different questions:

1. **A diagnostic panel.** A small set of markers, measured cheaply, that tells whether a patient has
   the disease. This is a *prediction* question.
2. **A list for the lab.** Every protein that is worth a follow-up experiment. This is a *discovery*
   question.
3. **A drug target.** One molecule that, if a drug changed it, would change the disease. This is a
   *causation* question.

### Task 1 · 3 questions, 3 judges

For each of the 3, write 3 things:

- the **goal**: prediction, discovery or causation;
- what the **result** looks like: a short set, a long list, 1 molecule;
- how you would **judge** whether the result is good.

For the 3rd, think about what kind of evidence would convince you that a molecule *causes* the
disease, rather than just travels with it. There is no code in this task, just write. Three short lines
per use is enough.

*Write your answer here.*

## B · A first look: rank everything

Most studies start the same way: test every feature on its own and sort. Let's do that, and then see
what the list is worth.

### Task 2 · Rank the 200 proteins

Rank the proteins by how well each one separates patients from controls on its own, and look at the
top 20. Then, in the cell after the code, write down **which of the top 20 you believe are
real**, and think about this: *why might a protein near the top of a list of 200 not be
real at all?*

In [ ]:
# TASK 2
# Think about: on its own, how strongly does each protein separate patients from controls?
# Try:         ranking = rank(df, PROTEINS)
#              ranking.head(20)



*Which of the top 20 do you believe are real? And why might some of them not be?*

### Before Task 3: a word about p-values when you test 200 things

A p-value of 0.05 means: *if this protein had nothing to do with the disease, a difference this large
would still show up 1 time in 20.* That is fine for 1 protein. But we tested 200. One
time in 20, over 200 tests, is about **10 proteins that look real and are not**, by
pure chance.

The `q` column is the fix. It is the p-value corrected for how many tests were run, and it has a simple
reading: **if you keep every protein with q ≤ 0.05, about 5% of what you keep is expected to be
chance.** This is called controlling the *false-discovery rate*. It does not make the list perfect; it
makes the error rate known, which is the honest version of a list.

### Task 3 · Keep an honest list, and get a second opinion

2 things. First, keep only the proteins that pass the correction (`passes`), and compare with what
you believed in Task 2. Second, run **stability selection** on the named markers plus the top 20
proteins, and look at how often the proteins that did *not* pass get selected. Do the 2 tools agree?

In [ ]:
# TASK 3
# Think about: which column of the ranking is the error rate? What does 'passes' promise?
# Try:         ranking[ranking.passes]
#              top20 = list(ranking.feature[:20])
#              freq = stability(df, MARKERS + top20)
#              freq[ [the proteins that did not pass] ]



**Your conclusion, in 1 sentence:** what did this part show?

## C · What a ranking cannot see

A ranking asks every protein the same question: *on your own, do you separate patients from
controls?* That misses 2 kinds of protein completely:

- one that carries no signal alone but a strong one **together with another feature**;
- one that looks real only because it **copies** a marker we already have.

The tool for both is the idea of **conditional independence**. 2 things are independent *given* a
third if, once you know the third, the first tells you nothing more about the second. Example: once you
know how much troponin a sample contains, does the hemolysis of the sample tell you anything more about
the disease? Yes, because hemolysis changes the troponin reading. And once you know LDL cholesterol,
does ApoB tell you anything more? Almost nothing: it is the same information.

The **Markov-boundary search** uses this to build a set, step by step: it keeps a feature if it still
tells you something about the disease once the already-selected features are known, and it drops a
feature that no longer does. Then it does 2 more things. It looks for features that say **nothing
alone but something together** with the set: a partner. And, for each selected feature, it lists the
features that could **replace** it. Those are the *statistically equivalent* features from the lecture,
the SES idea.

The search takes about 2 minutes on all 213 features. Start it, then read on.

### Task 4 · Run the search on everything

Run the Markov-boundary search on all the features: the named markers and the 200 proteins. Then
compare its list with the honest list from Task 3: what did the search **add** that the ranking never
saw, and what did it **drop**? Keep its equivalences; you need them in part D.

In [ ]:
# TASK 4
# Think about: a ranking asks each feature alone. What question is the search asking instead?
# Try:         selected, equivalences = markov_boundary(df, ALL)       (about 2 minutes)
#              honest_list = select(df, "filter", ALL)
#              [f for f in selected if f not in honest_list]
#              [f for f in honest_list if f not in selected]



### Task 5 · Nothing alone, something together

The search added 2 proteins that the ranking had ignored, and it added hemolysis. Take the 2
proteins. How well does each one separate patients from controls **alone**? And **together**? Then ask
the toolkit the precise question with `ci_test`: is the second protein linked to the disease on its own?
And given the 1st one? Do the same for hemolysis: alone, and given troponin.

Why does this happen in biology? Because biology often works through **balances, not single levels**:
a precursor and its product, an enzyme and its inhibitor, the free and the bound form of a hormone.
Each level alone can look unremarkable while the *ratio* carries the whole signal. A method that looks
at one feature at a time will never see it.

In [ ]:
# TASK 5
# Think about: if 2 proteins carry a signal only in their difference, what does each look like alone?
# Try:         auc_alone(df, f)       auc_together(df, [f, g])
#              ci_test(df, g)         ci_test(df, g, given=[f])
#              ci_test(df, "hemolysis")       ci_test(df, "hemolysis", given=["troponin"])



**Your conclusion, in 1 sentence:** what did this part show?

## D · 3 methods, 3 lists

Same data, 3 selection methods. They will not agree. The interesting part is *how* they disagree -
especially on 2 pairs of features that carry the same information.

### Task 6 · How does each method treat a pair of copies?

Run the **Lasso** on the named markers plus the top 20 proteins. Then look at 2 pairs: LDL
cholesterol and ApoB, and Lp(a) together with the protein the search listed as its equivalent in Task 4.
For each pair, check:

- which of the 2 the Lasso keeps;
- the **stability** frequency of each (from Task 3: `freq`);
- what the **search** said: one in the set, the other as its equivalent.

3 methods, 3 answers. Which one is right?

In [ ]:
# TASK 6
# Think about: two features carry the same information. What does a method that returns ONE list do with them?
# Try:         lasso_list = select(df, "lasso", MARKERS + top20)
#              copy_of_lpa = equivalences["lpa"][0]
#              freq[["ldl_c", "apob", "lpa", copy_of_lpa]]



**Your conclusion, in 1 sentence:** what did this part show?

## E · Predictive is not the same as causal

Now the best predictor in the study, and what it is worth.

One thing to know first. A **statin** is a drug that lowers LDL cholesterol. Almost every patient is
prescribed one at the moment they are diagnosed. The blood in this study was taken *after* diagnosis,
so the patients had already started their treatment when they were measured. Keep that in mind.

### Task 7 · The best predictor

Step by step:

1. Rank **all** features, named markers included. Which single feature separates patients from
   controls best? Look at both ends of the ranking. Remember that an AUC of 0.1 is as strong as 0.9.
2. Take the best *protein*, and also the strongest protein at the other end of the ranking. For each, ask:
   **within patients only, does it differ between those on a statin and those not?** A marker that does
   is tracking the treatment.
3. Now LDL cholesterol. What is its AUC? What is its link to the disease alone, and *given* statin use?
   Think about what the patients' statins do to their LDL.

In [ ]:
# TASK 7
# Think about: a marker written down AFTER the diagnosis can be a perfect predictor. How would you catch it?
# Try:         all_ranked = rank(df, ALL)
#              all_ranked.head(3)      all_ranked.tail(3)
#              within_patients(df, f, by="statin")
#              auc_alone(df, "ldl_c")      ci_test(df, "ldl_c")      ci_test(df, "ldl_c", given=["statin"])



**Your conclusion, in 1 sentence:** what did this part show?

## Your verdicts

### Task 8 · One line per protein

Now put it together. For **every protein you believe matters**: the ones that passed in Task 3, the
ones the search added, and anything else you are unsure about, write 1 row in the table below:

| protein | real, or chance? | kind of link | usable for | the evidence, 1 line |
|---|---|---|---|---|

**Kind of link**, from the lecture: *cause* · *effect of the disease* · *confounded* (shares a cause with
the disease) · *copy* of another marker · *consequence of treatment* · *pair* (only with a partner) ·
*artefact of the study* · *chance*.
**Usable for:** *panel* · *list* · *target* · *nothing*.

How to decide, for each protein: this is what the toolkit was for:

- **alone and together** (`auc_alone`, `auc_together`): nothing alone but strong with a partner → *pair*;
- **given the others** (`ci_test`, the search): drops out once a named marker is known → *copy* or
  *confounded*: a copy is tied to one marker, a confounded protein to BMI and smoking;
- **within patients, by treatment** (`within_patients`): differs between treated and untreated → *consequence*;

A protein that passes everything and is strongly linked to the disease is most likely an *effect*: the
disease produces it. We expect about 6 to 8 rows. Being unsure is fine; say what you would need.

| protein | real, or chance? | kind of link | usable for | the evidence, 1 line |
|---|---|---|---|---|
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |

In [ ]:
# the reveal: this cell is empty in your notebook; the instructor runs it
pass

## The report

3 lines, as you would write them in a thesis, then **upload this notebook to Moodle**.

**The panel I would propose, and how I would judge it:**

**The follow-up list, and its error rate:**

**The target, or "cannot be decided from these data", and why:**

In [ ]:
# scratch
